# PyEncode — `encode()` Demo
Demonstrates the primary API described in the paper.
All examples use `encode(VectorObj, N)` with the new constructors:
`SPARSE`, `STEP`, `SQUARE`, `FOURIER`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
from qiskit.quantum_info import Statevector

from pyencode import (
    encode,
    SPARSE, STEP, SQUARE, FOURIER,
)


## 1. SPARSE — Gleinig-Hoefler sparse state
`SPARSE([(x1, a1), (x2, a2), ...])` — $\mathcal{O}(s \cdot m)$ gates

s=1 reduces to a single basis state (X gates only).

In [ ]:
# s=1: basis vector at index 3
circuit, info = encode(SPARSE([(3, 5.0)]), N=8)
print(info)
circuit.draw("mpl")


In [ ]:
# Verify: all weight at index 3
sv = np.abs(np.array(Statevector(circuit)))
print("Statevector (abs):", np.round(sv, 4))


In [ ]:
# Gate count equals Hamming weight of index
for k in [0, 1, 3, 7, 19]:
    _, info = encode(SPARSE([(k, 1.0)]), N=64)
    print(f"k={k:>2} = {bin(k):>10}:  {info.gate_count} gates")


In [ ]:
# s=2: two point masses
circuit, info = encode(SPARSE([(1, 3.0), (6, 4.0)]), N=8)
print(info)
circuit.draw("mpl")


In [ ]:
# Verify: weight at indices 1 and 6 only
sv = np.abs(np.array(Statevector(circuit)))
print("Statevector (abs):", np.round(sv, 4))


In [ ]:
# s=5 point loads
entries = [(1, 1.0), (5, 2.0), (10, 3.0), (20, 1.5), (30, 0.5)]
circuit, info = encode(SPARSE(entries), N=32)
print(f"5 loads, N=32: {info.gate_count} gates, complexity={info.complexity}")


## 2. STEP — prefix uniform superposition
`STEP(k_s, c)` — $\mathcal{O}(m)$ gates (Shukla & Vedula 2024)

`STEP(k_s=N)` produces the uniform superposition $H^{\otimes m}|0\rangle$.

In [ ]:
circuit, info = encode(STEP(k_s=4, c=1.0), N=8)
print(info)
circuit.draw("mpl")


In [ ]:
# Verify: first 4 entries nonzero
sv = np.abs(np.array(Statevector(circuit)))
print("Statevector (abs):", np.round(sv, 4))


In [ ]:
# Non-power-of-2 boundary (general Shukla-Vedula construction)
circuit, info = encode(STEP(k_s=3, c=1.0), N=8)
print(f"k_s=3: gate count = {info.gate_count}")
circuit.draw("mpl")


In [ ]:
# Full range = uniform superposition
circuit, info = encode(STEP(k_s=8, c=1.0), N=8)
print(f"k_s=N: gate count = {info.gate_count}  (m Hadamard gates)")
circuit.draw("mpl")


## 3. SQUARE — general interval uniform superposition
`SQUARE(k1, k2, c)` — $\mathcal{O}(m)$ gates (this work)

Novel contribution extending STEP to arbitrary intervals $[k_1, k_2)$.

In [ ]:
circuit, info = encode(SQUARE(k1=2, k2=6, c=1.0), N=8)
print(info)
circuit.draw("mpl")


In [ ]:
# Verify: entries 2,3,4,5 nonzero
sv = np.abs(np.array(Statevector(circuit)))
print("Statevector (abs):", np.round(sv, 4))


In [ ]:
# Non-aligned square (general case)
circuit, info = encode(SQUARE(k1=3, k2=7, c=1.0), N=8)
print(f"Non-aligned [3,7): gate count = {info.gate_count}")
circuit.draw("mpl")


## 4. FOURIER — sinusoidal modes via inverse QFT
`FOURIER(modes=[(n, A, phi), ...])` — $\mathcal{O}(m^2)$ gates

T=1 mode: subsumes SINE (phi=0) and COSINE (phi=pi/2).

In [ ]:
# Single mode — sine (phi=0)
circuit, info = encode(FOURIER(modes=[(1, 1.0, 0)]), N=8)
print(info)
circuit.draw("mpl")


In [ ]:
# Verify against expected sine vector
sv = np.abs(np.array(Statevector(circuit)))
k = np.arange(8)
expected = np.sin(2 * np.pi * k / 8)
expected = np.abs(expected / np.linalg.norm(expected))
print("Circuit:  ", np.round(sv, 4))
print("Expected: ", np.round(expected, 4))


In [ ]:
# Single mode — cosine (phi=pi/2)
import math
circuit, info = encode(FOURIER(modes=[(1, 1.0, math.pi/2)]), N=8)
print(f"Cosine (phi=pi/2): gate count = {info.gate_count}")
circuit.draw("mpl")


In [ ]:
# Higher mode with phase
circuit, info = encode(FOURIER(modes=[(2, 1.0, math.pi/4)]), N=16)
print(f"n=2, phi=pi/4, N=16: gate count = {info.gate_count}")
circuit.draw("mpl")


In [ ]:
# Multi-mode T=2
circuit, info = encode(FOURIER(modes=[(1, 2.0, 0), (3, 1.0, 0)]), N=16)
print(info)
circuit.draw("mpl")


## 5. GEOMETRIC — exponential decay as product state
`GEOMETRIC(ratio, c)` — $\mathcal{O}(m)$ gates, **zero two-qubit gates**

The vector $f_i = c \cdot r^i$ is multiplicatively separable over the bits of $i$,
so the quantum state is a product state prepared by $m$ independent $R_y$ rotations.
See Xie & Ben-Ami (arXiv:2507.20317) for prior use.

In [ ]:
# Exponential decay: ratio = 0.5
circuit, info = encode(GEOMETRIC(ratio=0.5), N=8)
print(info)
circuit.draw("mpl")


In [ ]:
# Verify: amplitudes proportional to 0.5^i
sv = np.abs(np.array(Statevector(circuit)))
print("Statevector (abs):", np.round(sv, 4))
f = 0.5 ** np.arange(8)
print("Expected (norm'd):", np.round(f / np.linalg.norm(f), 4))


In [ ]:
# Gate count = m, two-qubit gates = 0 at every m
for m in [3, 4, 6, 8, 10]:
    _, info = encode(GEOMETRIC(ratio=0.95), N=2**m)
    print(f"m={m:2d}  gates={info.gate_count}  "
          f"1q={info.gate_count_1q}  2q={info.gate_count_2q}")


In [ ]:
# Growth: ratio > 1
circuit, info = encode(GEOMETRIC(ratio=2.0), N=8)
print(f"ratio=2.0: gates={info.gate_count}, 2q={info.gate_count_2q}")
circuit.draw("mpl")


## 6. Validation
`validate=True` simulates the circuit and checks the statevector.

In [ ]:
circuit, info = encode(FOURIER(modes=[(1, 1.0, 0)]), N=8, validate=True)
print(f"Validated: {info.validated}")


## 7. circuit_code
Every call returns a standalone Qiskit snippet in `info.circuit_code`.

In [ ]:
_, info = encode(SPARSE([(3, 1.0)]), N=8)
print(info.circuit_code)
